In [ ]:
# Cell 1: Install YOLOv8
!pip install ultralytics opencv-python numpy -q

import cv2
import numpy as np
from ultralytics import YOLO
from google.colab import files

print("YOLO26 installed and ready!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLO26 installed and ready!


In [ ]:
  # Cell 2: Trigonometry Function
def calculate_angle(a, b, c):
    a = np.array(a) # Shoulder
    b = np.array(b) # Elbow
    c = np.array(c) # Wrist

    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)

    if angle > 180.0:
        angle = 360 - angle

    return angle

print("Math functions loaded!")

Math functions loaded!


In [ ]:
# Cell 3: YOLO26-Pose (Large) with Complete Background Skeleton Isolation

# Load the highly accurate YOLO26 Large pose model
model = YOLO('yolo26l-pose.pt')

input_video_path = 'input_pitch.mp4'
output_video_path = 'output_pitcher_isolated_yolo26.mp4'

cap = cv2.VideoCapture(input_video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

# Define the standard 17-point skeleton connections (Same for YOLO26)
SKELETON_CONNECTIONS = [
    (0, 1), (0, 2), (1, 3), (2, 4),       # Face lines
    (5, 6),                               # Shoulders
    (5, 7), (7, 9),                       # Left Arm
    (6, 8), (8, 10),                      # Right Arm
    (5, 11), (6, 12), (11, 12),           # Torso
    (11, 13), (13, 15),                   # Left Leg
    (12, 14), (14, 16)                    # Right Leg
]

print("Downloading YOLO26l weights and isolating pitcher... please wait.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLO26 inference on the frame
    results = model(frame, verbose=False)

    for result in results:
        # Check if the model detected anyone and mapped their boxes
        if result.keypoints is not None and len(result.keypoints.xy) > 0 and result.boxes is not None:

            # 1. Calculate the area of all bounding boxes to find the main pitcher
            boxes = result.boxes.xyxy.cpu().numpy()
            areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
            largest_person_idx = np.argmax(areas)

            # 2. Extract keypoints ONLY for the largest person (the pitcher)
            keypoints = result.keypoints.xy[largest_person_idx].cpu().numpy()

            # 3. Draw the skeleton connections ONLY for this person
            for kp1_idx, kp2_idx in SKELETON_CONNECTIONS:
                pt1 = keypoints[kp1_idx]
                pt2 = keypoints[kp2_idx]

                # Verify both points were detected properly by the model
                if pt1[0] > 0 and pt2[0] > 0:
                    cv2.line(frame, (int(pt1[0]), int(pt1[1])), (int(pt2[0]), int(pt2[1])), (255, 0, 255), 3)

            # 4. Draw the joint dots ONLY for this person
            for idx, pt in enumerate(keypoints):
                if pt[0] > 0:
                    cv2.circle(frame, (int(pt[0]), int(pt[1])), 5, (0, 255, 255), -1)

            # 5. Calculate and overlay the right elbow angle
            try:
                # YOLO Indices: 6=R_Shoulder, 8=R_Elbow, 10=R_Wrist
                r_shoulder = keypoints[6]
                r_elbow = keypoints[8]
                r_wrist = keypoints[10]

                if all(p[0] > 0 for p in [r_shoulder, r_elbow, r_wrist]):
                    angle = calculate_angle(r_shoulder, r_elbow, r_wrist)

                    # Text overlay for the angle metric
                    cv2.putText(frame, f"{int(angle)} Deg",
                                (int(r_elbow[0]) + 15, int(r_elbow[1])),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3, cv2.LINE_AA)
            except Exception as e:
                pass

    # Write the frame directly (No .plot() means no background skeletons)
    out.write(frame)

cap.release()
out.release()
print("Processing complete! Download the new video.")

Processing complete! Download the new video.
